In [11]:
import pandas as pd

In [12]:
# To filter the list of interactions to only those which present an adverse reaction..
# 1) Obtain the list of interactions from the DDI dataset.
# 2) Extract the templates by replacing drug names with a placeholder token
#   --> DrugBank's interaction descriptions are template-generated, NOT FREE-FORM TEXT.
#   --> There is a small, finite set of sentence patterns reused across the millions of interactions 
# 3) Then use set() or simialar operation to obtain the unique set of templates.
# 4) Once we have the templates we can classify each template into severity tiers (or "unknown" if we can't determine it).
# 5) Then we can filter the interactions to only those which are labeled as "adverse reaction" templates.

drug_bank_csv = r"C:\Users\ashto\ddi-prediction\data\sample/drugbank_parsed_with_interactions.csv"  # Path to the CSV file containing parsed DrugBank data

main_df = pd.read_csv(drug_bank_csv)  # Load the CSV file into a DataFrame

main_df.head()  # Display the first few rows of the DataFrame to verify it loaded correctly


,drugbank_id,name,smiles,inchi,groups,drug_type,atc_codes,interactions,n_interactions
0,DB00006,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,InChI=1S/C98H138N24O33/c1-5-52(4)82(96(153)122...,"['approved', 'investigational']",small molecule,['B01AE06'],"[{'drugbank_id': 'DB06605', 'name': 'Apixaban'...",641
1,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,InChI=1S/C59H84N18O14/c1-31(2)22-40(49(82)68-3...,"['approved', 'investigational']",small molecule,['L02AE03'],"[{'drugbank_id': 'DB09066', 'name': 'Corifolli...",1156
2,DB00027,Gramicidin D,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,InChI=1S/C96H135N19O16/c1-50(2)36-71(105-79(11...,"['approved', 'investigational']",small molecule,['R02AB30'],"[{'drugbank_id': 'DB12768', 'name': 'BCG vacci...",56
3,DB00035,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,InChI=1S/C46H64N14O12S2/c47-35(62)15-14-29-40(...,"['approved', 'investigational']",small molecule,['H01BA02'],"[{'drugbank_id': 'DB00564', 'name': 'Carbamaze...",1140
4,DB00080,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,InChI=1S/C72H101N17O26/c1-5-6-7-8-9-10-11-22-5...,"['approved', 'investigational']",small molecule,['J01XX09'],"[{'drugbank_id': 'DB00243', 'name': 'Ranolazin...",1049


In [13]:
interaction_descriptions = main_df['interactions'].tolist()  # Extract the interaction descriptions into a list

In [32]:
import ast
import re

def templatize_description(desc, drug1_name, drug2_name):
    """Replace drug names with {DRUG} using exact match first, then regex fallback."""
    # Step 1: Replace known drug names (longest first to avoid partial matches)
    names = sorted([drug1_name, drug2_name], key=len, reverse=True)
    template = desc
    for name in names:
        template = template.replace(name, "{DRUG}")

    # Step 2: Regex-based cleanup for any remaining drug/metabolite names in known sentence slots

    # "The serum concentration of X, an active metabolite of Y, can be ... when used in combination with Z."
    template = re.sub(
        r'The serum concentration of (.+?), an active metabolite of (.+?), can be (increased|decreased) when used in combination with (.+?)\.',
        r'The serum concentration of {METABOLITE}, an active metabolite of {DRUG}, can be \3 when used in combination with {DRUG}.', template)

    # "The serum concentration of the active metabolites of X can be ... when Y is used in combination with Z (resulting in ...)."
    template = re.sub(
        r'The serum concentration of the active metabolites of (.+?) can be (increased|decreased|reduced) when (.+?) is used in combination with (.+?)( resulting in .+?)?\.',
        r'The serum concentration of the active metabolites of {DRUG} can be \2 when {DRUG} is used in combination with {DRUG}.', template)

    # "The serum concentration of X can be ... when Y is combined with Z."
    template = re.sub(
        r'The serum concentration of (.+?) can be (increased|decreased) when (.+?) is combined with (.+?)\.',
        r'The serum concentration of {DRUG} can be \2 when {DRUG} is combined with {DRUG}.', template)

    # "The serum concentration of X can be ... when it is combined with Y."
    template = re.sub(
        r'The serum concentration of (.+?) can be (increased|decreased) when it is combined with (.+?)\.',
        r'The serum concentration of {DRUG} can be \2 when it is combined with {DRUG}.', template)

    # "The metabolism of X can be ... when combined with Y."
    template = re.sub(
        r'The metabolism of (.+?) can be (increased|decreased) when combined with (.+?)\.',
        r'The metabolism of {DRUG} can be \2 when combined with {DRUG}.', template)

    # "The risk or severity of X can be ... when Y is combined with Z."
    template = re.sub(
        r'when (.+?) is combined with (.+?)\.',
        r'when {DRUG} is combined with {DRUG}.', template)

    # "The therapeutic efficacy of X can be ... when used in combination with Y."
    template = re.sub(
        r'The therapeutic efficacy of (.+?) can be (increased|decreased) when used in combination with (.+?)\.',
        r'The therapeutic efficacy of {DRUG} can be \2 when used in combination with {DRUG}.', template)

    # "The excretion of X can be ... when combined with Y."
    template = re.sub(
        r'The excretion of (.+?) can be (increased|decreased) when combined with (.+?)\.',
        r'The excretion of {DRUG} can be \2 when combined with {DRUG}.', template)

    # "X may increase/decrease the ... activities of Y."
    template = re.sub(
        r'^(.+?) may (increase|decrease) the (.+?) activities of (.+?)\.',
        r'{DRUG} may \2 the \3 activities of {DRUG}.', template)

    # "X may decrease effectiveness of Y as a diagnostic agent."
    template = re.sub(
        r'^(.+?) may decrease effectiveness of (.+?) as a diagnostic agent\.',
        r'{DRUG} may decrease effectiveness of {DRUG} as a diagnostic agent.', template)

    # "X may increase/decrease the excretion rate of Y which could..."
    template = re.sub(
        r'^(.+?) may (increase|decrease) the excretion rate of (.+?) which could',
        r'{DRUG} may \2 the excretion rate of {DRUG} which could', template)

    # "X can cause a decrease/increase in the absorption of Y resulting in..."
    template = re.sub(
        r'^(.+?) can cause an? (decrease|increase) in the absorption of (.+?) resulting in',
        r'{DRUG} can cause a \2 in the absorption of {DRUG} resulting in', template)

    return template

def create_template_list(main_df):
    template_set = set()

    for i in range(len(main_df)):
        drug1_name = main_df['name'].iloc[i]
        interactions = ast.literal_eval(main_df['interactions'].iloc[i])

        for interaction in interactions:
            drug_2_name = interaction['name']
            desc = interaction['description']
            template = templatize_description(desc, drug1_name, drug_2_name)
            template_set.add(template)

    return template_set

templates = create_template_list(main_df)  # Process ALL rows to get every unique template
print(f"Number of unique templates: {len(templates)}\n")
for t in sorted(templates):
    print(t)

Number of unique templates: 307

The absorption of {DRUG} can be decreased when combined with {DRUG}.
The bioavailability of {DRUG} can be decreased when combined with {DRUG}.
The bioavailability of {DRUG} can be increased when combined with {DRUG}.
The excretion of {DRUG} can be decreased when combined with {DRUG}.
The excretion of {DRUG} can be increased when combined with {DRUG}.
The metabolism of {DRUG} can be decreased when combined with {DRUG}.
The metabolism of {DRUG} can be increased when combined with {DRUG}.
The protein binding of {DRUG} can be decreased when combined with {DRUG}.
The protein binding of {DRUG} can be increased when combined with {DRUG}.
The risk of a hypersensitivity reaction to {DRUG} is increased when {DRUG} is combined with {DRUG}.
The risk or severity of Anticonvulsant Toxicity can be increased when {DRUG} is combined with {DRUG}.
The risk or severity of CNS depression and hypotonia can be increased when {DRUG} is combined with {DRUG}.
The risk or severit

In [33]:
# Find templates that still contain text outside {DRUG}/{METABOLITE} placeholders that looks like an unprocessed drug name
missed = [t for t in templates if '{DRUG}' in t and re.search(
    r'(?<!\{)(?:combined with|activities of|excretion rate of|efficacy of|concentration of|excretion of|metabolism of) (?!\{DRUG\}|\{METABOLITE\}|the |it )(\S+)', t)]
print(f"Templates with possible leaked drug names: {len(missed)}\n")
for t in sorted(missed):
    print(t)

Templates with possible leaked drug names: 0



## Classifying the drug descriptions

In [ ]:
# Group 1: Explicity adverse reactions (e.g. "The risk or severity of X can be ... when Y is combined with Z.")
# Group 2: Protective / reduced-risk interactions 